# 01 — DANI audit, MPS training, and validation-only selection

This notebook reconstructs the academic decision path from saved artifacts. It verifies the canonical DANI data gate, summarises the matched RGB/FFT experiments over seeds 7, 17, and 42, and checks that the final representation and representative checkpoint were selected **without internal-test metrics**. Training is not repeated automatically; the exact launch contract is read from the run artifacts.

In [1]:
import json
from pathlib import Path

import pandas as pd
import torch


def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src/ai_image_detector').is_dir():
            return candidate
    raise RuntimeError('Run the notebook from this repository or a child directory.')


REPO = find_repository_root()
print('Repository:', REPO)
print('PyTorch:', torch.__version__)
print('MPS available in this process:', torch.backends.mps.is_available())

Repository: /Users/robertarifulin/Documents/ChatGPT/AI Image Detector
PyTorch: 2.13.0
MPS available in this process: True


## 1. Canonical dataset gate

The source audit found label-dependent container/mode support in the downloaded files. Therefore the neural study uses a losslessly canonicalised 1024×1024 RGB PNG corpus. A parent group contains one COCO photograph and four generated conditions and is assigned wholly to train, validation, or test.

In [2]:
AUDIT = REPO / 'artifacts/audits/dani_rgb1024_integrity_v1/summary.json'
MANIFEST = REPO / 'artifacts/audits/dani_rgb1024_integrity_v1/training_manifest.csv'
audit = json.loads(AUDIT.read_text(encoding='utf-8'))
manifest = pd.read_csv(MANIFEST)

assert audit['eligibility']['eligible_for_training'] is True
assert audit['counts']['cross_label_exact_duplicate_group_count'] == 0
assert audit['counts']['cross_split_integrity_component_count'] == 0
assert audit['shortcut_audit']['format_mode_support_balanced_between_labels'] is True
assert len(manifest) == 7410 and manifest['group_id'].nunique() == 1482
assert not manifest.groupby('group_id')['split'].nunique().gt(1).any()
display(manifest.groupby(['split', 'generator']).size().rename('images').unstack(fill_value=0))
pd.Series({
    'rows': len(manifest),
    'parent_groups': manifest['group_id'].nunique(),
    'manifest_sha256': audit['training_manifest_sha256'],
    'eligible_for_training': audit['eligibility']['eligible_for_training'],
})

generator,Dalle3:T2I,SD_XL:I2I,SD_XL:T2I,SD_XL:TI2I,real
split,,,,,
test,222,222,222,222,222
train,1037,1037,1037,1037,1037
val,223,223,223,223,223


rows                                                                  7410
parent_groups                                                         1482
manifest_sha256          3b88920920add51ef8c55b225817448b759c05a8d5bfc1...
eligible_for_training                                                 True
dtype: object

## 2. Validation-only controls and matched neural queue

The metadata control measures acquisition-pipeline shortcuts; it is not a detector. The radial logistic baseline sees only 64 radial FFT bins. The primary comparison uses ResNet-50 from random initialisation for both RGB and FFT, a common 384×384 raster, batch 64, learning rate 10⁻⁴, at most 15 epochs, patience 4, parent-paired sampling, and Apple MPS. Every run writes validation predictions only.

In [3]:
def read_metrics(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

control_paths = {
    'original metadata': REPO / 'artifacts/baselines/dani_source_shortcut_validation_v1/file_metadata_control_seed7/validation_selection_metrics.json',
    'canonical metadata': REPO / 'artifacts/baselines/dani_canonical_metadata_validation_v1/file_metadata_control_seed7/validation_selection_metrics.json',
    'radial FFT logistic': REPO / 'artifacts/baselines/dani_radial_fft_validation_v1/radial_fft_logistic_highres_square_crop_384_v1_seed7/validation_selection_metrics.json',
}
controls = pd.DataFrame.from_dict({name: read_metrics(path) for name, path in control_paths.items()}, orient='index')
display(controls[['roc_auc', 'balanced_accuracy', 'macro_f1', 'fpr_at_tpr_95', 'ece_15']])

rows = []
for representation in ('rgb', 'fft'):
    for seed in (7, 17, 42):
        run_dir = REPO / f'artifacts/experiments/dani_{representation}_scratch_seed{seed}_validation_v1'
        assert (run_dir / 'validation_metrics.json').is_file()
        assert not (run_dir / 'internal_test_metrics.json').exists()
        rows.append({'representation': representation, 'seed': seed, **read_metrics(run_dir / 'validation_metrics.json')})
validation_runs = pd.DataFrame(rows)
display(validation_runs[['representation', 'seed', 'roc_auc', 'balanced_accuracy', 'macro_f1', 'fpr_at_tpr_95']])

,roc_auc,balanced_accuracy,macro_f1,fpr_at_tpr_95,ece_15
original metadata,0.713427,0.660314,0.552612,0.887892,0.246208
canonical metadata,0.537373,0.573430,0.509027,0.991031,0.297057
radial FFT logistic,0.862972,0.803251,0.810515,0.376682,0.170572


,representation,seed,roc_auc,balanced_accuracy,macro_f1,fpr_at_tpr_95
0,rgb,7,0.732917,0.675448,0.601795,0.829596
1,rgb,17,0.739031,0.692265,0.584206,0.843049
2,rgb,42,0.633946,0.603700,0.472148,0.883408
3,fft,7,0.922203,0.843049,0.796153,0.336323
4,fft,17,0.897454,0.823430,0.772210,0.394619
5,fft,42,0.736341,0.696188,0.573436,0.843049


## 3. Frozen validation decision

Representation selection uses mean validation balanced accuracy across all three seeds. Exact ties would be resolved by mean validation ROC-AUC and then the representation name. Seed 17 was declared in advance as the representative checkpoint; the numerically strongest seed is not selected.

In [4]:
AGGREGATE = REPO / 'artifacts/aggregates/dani_rgb_fft_scratch_validation_v1/aggregate_summary.json'
SELECTION = REPO / 'artifacts/aggregates/dani_rgb_fft_scratch_validation_v1/prototype_selection_record.json'
aggregate = read_metrics(AGGREGATE)
selection = read_metrics(SELECTION)

assert aggregate['status'] == 'validation_aggregated'
assert selection['selection_status'] == 'validation_selected_pending_external_validation'
assert selection['selection_rule']['internal_test_metrics_used_for_selection'] is False
assert set(selection['selection_rule']['expected_seeds']) == {7, 17, 42}
assert set(selection['selection_rule']['expected_representations']) == {'rgb', 'fft'}
assert selection['decision']['representative_seed'] == 17

summary_rows = []
for representation, metrics in aggregate['validation_metrics_by_representation'].items():
    summary_rows.append({
        'representation': representation,
        **{f'{name}_mean': values['mean'] for name, values in metrics.items()},
        **{f'{name}_sd': values['std'] for name, values in metrics.items()},
    })
display(pd.DataFrame(summary_rows).set_index('representation'))
pd.Series(selection['decision'])

,balanced_accuracy_mean,fpr_at_tpr_95_mean,macro_f1_mean,roc_auc_mean,balanced_accuracy_sd,fpr_at_tpr_95_sd,macro_f1_sd,roc_auc_sd
representation,,,,,,,,
fft,0.787556,0.524664,0.713933,0.852000,0.079732,0.277266,0.122262,0.100925
rgb,0.657138,0.852018,0.552717,0.701965,0.047036,0.028004,0.070326,0.058985


checkpoint_sha256          0600e1e3346058ad826f61f37515022aad275b221cf9e7...
experiment                             dani_fft_scratch_seed17_validation_v1
experiment_dir             /Users/robertarifulin/Documents/ChatGPT/AI Ima...
made                                                                    True
reason                     highest mean validation balanced_accuracy; exa...
representative_seed                                                       17
selected_representation                                                  fft
validation_metrics         {'balanced_accuracy': 0.8234304932735426, 'fpr...
validation_threshold                                                0.822677
dtype: object

### Interpretation

FFT is the validation-selected representation under the matched-from-scratch protocol (mean BAcc 0.7876 versus 0.6571 for RGB), but the FFT standard deviations are large. This is evidence about the declared DANI distribution, not universal provenance. The next notebook may read only the immutable terminal evaluation created after this record. External Synthbuster + RAISE evaluation remains unavailable and is reported as a limitation.